In [ ]:
# ===============================
# Check label distribution
# ===============================

df_labeled.groupBy('retained').count().show()

In [ ]:
# ===============================
# Select basic model features
# ===============================

feature_cols = [
    'author_num_games_owned',
    'author_num_reviews',
    'author_playtime_forever',
    'author_playtime_at_review',
    'votes_up',
    'comment_count',
    'weighted_vote_score'
]

df_model = (
    df_labeled
    .select(feature_cols + ['retained'])
)

In [ ]:
# ===============================
# Assemble features
# ===============================

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='finalized_features'
)

df_features = (
    assembler
    .transform(df_model)
    .select('features', 'retained')
)

### Retention / Churn Heuristic

For this initial baseline model, a player is considered **retained** if they recorded more than 60 minutes of playtime within the last two weeks. Players with 60 minutes or less are treated as likely churned users.

The 60-minute threshold acts as apractical behavioral heuristic rather than a definitive business rule. The assumption is that players who return and spend at least one hour actively engaging with a game over a recent two-week period demonstrate meaningful continued interest and engagement. In contrast, very low or zero recent playtime may indicate disengagement, abandonment, or temporary inactivity.

This threshold was intentionally chosen as a lightweight and interpretable starting point for experimentation. It helps transform continuous playtime behavior into a binary classification problem suitable for Logistic Regression while remaining easy to explain from a business perspective.

Future iterations of the project may refine this definition using:
- percentile-based engagement thresholds,
- genre-specific activity expectations,
- rolling activity windows,
- survival analysis,
- or clustering methods to identify natural retention breakpoints.

In [ ]:
# ===============================
# Define churn / retention label
# ===============================
# retained = 1 means the player came back
# retained = 0 means likely churned

df_labeled = (
    df_english
    .filter(F.col('author_playtime_last_two_weeks').isNotNull())
    .withColumn(
        'retained',
        F.when(F.col('author_playtime_last_two_weeks') > 60, 1).otherwise(0)
    )
)

In [ ]:
def add_derived_columns(df: DataFrame) -> DataFrame:
    """
    Mirror the spirit of your Pandas PreEDA:
    - convert integer epoch columns to timestamps
    - add review_length
    - add a few quality flags
    - add log1p helper columns for skewed counts
    """
    out = df

    # Timestamp conversions
    out = out.withColumn("author_last_played_ts", F.to_timestamp(F.from_unixtime("author_last_played")))
    out = out.withColumn("timestamp_created_ts", F.to_timestamp(F.from_unixtime("timestamp_created")))
    out = out.withColumn("timestamp_updated_ts", F.to_timestamp(F.from_unixtime("timestamp_updated")))

    # Text length
    out = out.withColumn(
        "review_length",
        F.when(F.col("review").isNull(), None).otherwise(F.length("review"))
    )
    
    # Mild derived features aligned to your sample notebook ideas
    for src, dst in [
        ("votes_up", "log_votes_up"),
        ("votes_funny", "log_votes_funny"),
        ("comment_count", "log_comment_count"),
        ("author_num_reviews", "log_author_num_reviews"),
        ("author_num_games_owned", "log_author_num_games_owned"),
        ("author_playtime_forever", "log_author_playtime_forever"),
        ("author_playtime_last_two_weeks", "log_author_playtime_last_two_weeks"),
        ("author_playtime_at_review", "log_author_playtime_at_review"),
        ("review_length", "log_review_length"),
    ]:
        out = out.withColumn(
            dst,
            F.when(F.col(src).isNull(), None).otherwise(F.log1p(F.col(src)))
        )

    return out



In [ ]:
# Consistency & Validity cleaning
def clean_playtime_consistency(df: DataFrame, playtime_cols: Iterable[str] = PLAYTIME_COLS, strategy: str = "flag",) -> DataFrame:
    """
    Handle rows where:
      - author_playtime_at_review > author_playtime_forever
      - author_playtime_last_two_weeks > author_playtime_forever

    Default is flag instead of drop because these can reflect API timing issues.
    """

    # Ensure columns exist in the input df
    if not set(playtime_cols).issubset(set(df.columns)):
        return df

    # Find inconsistent instances (current playtime < all playtime)
    at_review_bad = (
        F.col("author_playtime_at_review").isNotNull()
        & F.col("author_playtime_forever").isNotNull()
        & (
            F.col("author_playtime_at_review") > 
            F.col("author_playtime_forever")
        )
    )
    two_week_bad = (
        F.col("author_playtime_last_two_weeks").isNotNull()
        & F.col("author_playtime_forever").isNotNull()
        & (
            F.col("author_playtime_last_two_weeks") > 
            F.col("author_playtime_forever")
        )
    )

    # Keep it as useful columns, used for feature engineering
    if strategy == "flag":
        return (
            df.withColumn(
                "flag_playtime_at_review_gt_forever",
                at_review_bad.cast("boolean")
            )
            .withColumn(
                "flag_playtime_2w_gt_forever",
                two_week_bad.cast("boolean")
            )
        )

    raise ValueError("strategy must be 'flag' or 'drop'")

def clean_negative_count_values(df: DataFrame, count_cols: Iterable[str] = NUMERIC_COUNT_COLS, strategy: str = "flag") -> DataFrame:
    """
    Handle negative values in count-like columns.

    strategy:
      - "null": replace invalid negative values with null
      - "drop": remove rows containing any negative count value
    """

    # Ensure columns exist in the input df
    cols = [
        c 
        for c in count_cols 
        if c in df.columns
    ]

    # Initialize the binary mask as all ones vector
    condition = F.lit(True)

    # If one row has at least one column with negative count, you return that row as 0
    for c in cols:
        condition = condition & (
            F.col(c).isNull() | 
            (F.col(c) >= 0)
        )
    
    # Keep it as useful columns, used for feature engineering
    if strategy == "flag":
        for c in cols:
            df = df.withColumn(
                f"flag_{c}_negative",
                (
                    F.col(c).isNotNull() &
                    (F.col(c) < 0)
                ).cast("boolean")
            )
        return df

    raise ValueError("strategy must be 'flag' or 'drop'")

def clean_weighted_vote_score(df: DataFrame, strategy: str = "flag") -> DataFrame:
    """
    Ensure weighted_vote_score is in [0, 1].
    """

    # Ensure columns exist in the input df
    c = "weighted_vote_score"
    if c not in df.columns:
        return df

    # Find invalid weighted score outside of range [0.0, 1.0]
    invalid = (
        F.col(c).isNotNull() &
        ~F.col(c).between(0.0, 1.0)
    )
    
    # Keep it as useful columns, used for feature engineering
    if strategy == "flag":
        return df.withColumn(
            "flag_weighted_vote_score_invalid",
            invalid.cast("boolean")
        )
 
    raise ValueError("strategy must be 'flag' or 'drop'")

def clean_impossible_two_week_playtime(df: DataFrame, strategy: str = "flag") -> DataFrame:
        """
    Handle author_playtime_last_two_weeks > 14 days of minutes.

    Default is flag, because near-limit values may reflect Steam runtime,
    not necessarily active human gameplay.
    """

    # Ensure columns exist in the input df
    c = "author_playtime_last_two_weeks"
    if c not in df.columns:
        return df

    # Find invalid instances where playtime > 2 weeks
    is_impossible = F.col(c).isNotNull() & (F.col(c) > F.lit(MAX_2W_MINUTES))
    
    # Keep it as useful columns, used for feature engineering
    if strategy == "flag":
        return df.withColumn(
            "flag_playtime_2w_exceeds_physical_limit",
            is_impossible.cast("boolean")
        )

    raise ValueError("strategy must be 'flag' or 'drop'")

In [ ]:
days_since_last_play
recency
feature cutoff T
label window

In [ ]:
lowercasing
URL replacement
emoji handling
punctuation normalization
accent normalization
contraction handling
tokenization
stopword removal
stemming / lemmatization
language-specific processing

TF-IDF
word n-grams
character n-grams
sentiment scores
embeddings
language-specific vectorizers
review length features